In [1]:
import math
from types import SimpleNamespace
from iron_data import IRON

In [2]:
# --- FACTORY PARAMETERS ---
TARGET_MODULAR_FRAMES = 4  # How many frames per minute do you want?

# Overclocking (1.0 = 100%, 2.5 = 250%)
# This doesn't change raw materials needed, just the number of machines!
MACHINE_CLOCK_SPEED = 1.0

In [3]:
# 1. Base Machine Ratios
# If we need 10 frames, and 1 machine makes 2/min, we need 5 base machines.
base_frame_assemblers = TARGET_MODULAR_FRAMES / IRON.MODULAR_FRAME.build_rate

# 2. Actual Buildings to Build (Factoring in overclocking)
actual_frame_assemblers = math.ceil(base_frame_assemblers / MACHINE_CLOCK_SPEED)

# 3. Items required per minute to feed these assemblers
req_reinforced_plates = base_frame_assemblers * IRON.MODULAR_FRAME.requirements.reinforced_plate
req_rods_for_frames = base_frame_assemblers * IRON.MODULAR_FRAME.requirements.rod

print(f"To make {TARGET_MODULAR_FRAMES} Modular Frames/min:")
print(f"--> Build {actual_frame_assemblers} Assemblers (Modular Frames)")
print(f"--> Requires {req_reinforced_plates} Reinforced Plates/min")
print(f"--> Requires {req_rods_for_frames} Rods/min (Just for the frames)")

To make 4 Modular Frames/min:
--> Build 2 Assemblers (Modular Frames)
--> Requires 6.0 Reinforced Plates/min
--> Requires 24.0 Rods/min (Just for the frames)


In [4]:
base_rip_assemblers = req_reinforced_plates / IRON.REINFORCED_PLATE.build_rate
actual_rip_assemblers = math.ceil(base_rip_assemblers / MACHINE_CLOCK_SPEED)

req_plates = base_rip_assemblers * IRON.REINFORCED_PLATE.requirements.plate
req_screws = base_rip_assemblers * IRON.REINFORCED_PLATE.requirements.screw

print(f"To feed those {req_reinforced_plates} RIPs/min:")
print(f"--> Build {actual_rip_assemblers} Assemblers (Reinforced Plates)")
print(f"--> Requires {req_plates} Iron Plates/min")
print(f"--> Requires {req_screws} Screws/min")

To feed those 6.0 RIPs/min:
--> Build 2 Assemblers (Reinforced Plates)
--> Requires 36.0 Iron Plates/min
--> Requires 72.0 Screws/min


In [5]:
base_screw_constructors = req_screws / IRON.SCREW.build_rate
actual_screw_constructors = math.ceil(base_screw_constructors / MACHINE_CLOCK_SPEED)

req_rods_for_screws = base_screw_constructors * IRON.SCREW.requirements.rod

print(f"To feed those {req_screws} Screws/min:")
print(f"--> Build {actual_screw_constructors} Constructors (Screws)")
print(f"--> Requires {req_rods_for_screws} Rods/min (Just for the screws)")

To feed those 72.0 Screws/min:
--> Build 2 Constructors (Screws)
--> Requires 18.0 Rods/min (Just for the screws)


In [6]:
# Calculate Total Rods
total_req_rods = req_rods_for_frames + req_rods_for_screws
base_rod_constructors = total_req_rods / IRON.ROD.build_rate
actual_rod_constructors = math.ceil(base_rod_constructors / MACHINE_CLOCK_SPEED)
req_ingots_for_rods = base_rod_constructors * IRON.ROD.requirements.ingot

# Calculate Total Plates
base_plate_constructors = req_plates / IRON.PLATE.build_rate
actual_plate_constructors = math.ceil(base_plate_constructors / MACHINE_CLOCK_SPEED)
req_ingots_for_plates = base_plate_constructors * IRON.PLATE.requirements.ingot

# Total Ingots
total_req_ingots = req_ingots_for_rods + req_ingots_for_plates

print(f"To feed {req_plates} Plates/min and {total_req_rods} Total Rods/min:")
print(f"--> Build {actual_plate_constructors} Constructors (Plates)")
print(f"--> Build {actual_rod_constructors} Constructors (Rods)")
print(f"--> Requires {total_req_ingots} Iron Ingots/min")

To feed 36.0 Plates/min and 42.0 Total Rods/min:
--> Build 2 Constructors (Plates)
--> Build 3 Constructors (Rods)
--> Requires 96.0 Iron Ingots/min


In [7]:
# Smelters
base_smelters = total_req_ingots / IRON.INGOT.build_rate
actual_smelters = math.ceil(base_smelters / MACHINE_CLOCK_SPEED)
total_req_ore = base_smelters * IRON.INGOT.requirements.ore

# Miners
base_miners = total_req_ore / IRON.ORE.build_rate
actual_miners = math.ceil(base_miners / MACHINE_CLOCK_SPEED)

print(f"To feed {total_req_ingots} Ingots/min:")
print(f"--> Build {actual_smelters} Smelters")
print(f"--> Requires {total_req_ore} Raw Iron Ore/min")
print(f"--> Build {actual_miners} Miners (Mk.1 on Normal Nodes)")

To feed 96.0 Ingots/min:
--> Build 4 Smelters
--> Requires 96.0 Raw Iron Ore/min
--> Build 2 Miners (Mk.1 on Normal Nodes)


In [8]:
# --- CALCULATE SPACE ELEVATOR PROJECT PARTS (TIER 2) ---
from tier2_project_parts import ProjectPartsCalculator

# Initialize the calculator for Tier 2 and Space Elevator parts
calc = ProjectPartsCalculator(clock_speed=1.0)

# Calculate requirements for 2 Smart Plating per minute
target_part = "smart_plating"
target_rate = 2.0

print(f"--- Production Plan for {target_rate} {target_part.replace('_', ' ').title()}/min ---")
print(calc.render_tree(target_part, target_rate))

results = calc.solve(target_part, target_rate)
print("\nRequired Raw Materials:")
for resource, rate in results["raw"].items():
    print(f" - {resource.replace('_', ' ').title()}: {rate:,.1f} units/min")

print("\nMachines Needed:")
for item, count in results["machines"].items():
    print(f" - {count}x {item.replace('_', ' ').title()}")


--- Production Plan for 2.0 Smart Plating/min ---
🎯 Smart Plating (2.0/min)  [1x Assembler @ 100%]
   ├─ Reinforced Iron Plate (2.0/min)  [1x Assembler @ 100%]
   │  ├─ Iron Plate (12.0/min)  [1x Constructor @ 100%]
   │  │  └─ Iron Ingot (18.0/min)  [1x Smelter @ 100%]
   │  │     └─ Iron Ore (18.0/min)  (Raw Resource)
   │  └─ Screw (24.0/min)  [1x Constructor @ 100%]
   │     └─ Iron Rod (6.0/min)  [1x Constructor @ 100%]
   │        └─ Iron Ingot (6.0/min)  [1x Smelter @ 100%]
   │           └─ Iron Ore (6.0/min)  (Raw Resource)
   └─ Rotor (2.0/min)  [1x Assembler @ 100%]
      ├─ Iron Rod (10.0/min)  [1x Constructor @ 100%]
      │  └─ Iron Ingot (10.0/min)  [1x Smelter @ 100%]
      │     └─ Iron Ore (10.0/min)  (Raw Resource)
      └─ Screw (50.0/min)  [2x Constructor @ 100%]
         └─ Iron Rod (12.5/min)  [1x Constructor @ 100%]
            └─ Iron Ingot (12.5/min)  [1x Smelter @ 100%]
               └─ Iron Ore (12.5/min)  (Raw Resource)

Required Raw Materials:
 - Iron Ore